 **complete setup document** for deploying a Flask + Iris model app to **Azure App Service using GitHub**, starting from scratch — **including Git repo creation, virtual environment, local testing, and Azure deployment.**

---

#  Deploy Flask + Iris Model App to Azure (GitHub-Based)

---

##  Prerequisites

1. Azure account: [https://portal.azure.com](https://portal.azure.com)
2. GitHub account and access to create repositories
3. Python installed (3.10+ recommended)
4. `git` and `pip` available on system
5. VS Code or preferred editor

---

##  Project Setup (Local)

### 1. Create Project Folder

```bash
mkdir iris-flask-app
cd iris-flask-app
```

### 2. Create & Activate Virtual Environment

```bash
python -m venv venv

# Activate:
# Windows:
venv\Scripts\activate

# macOS/Linux:
source venv/bin/activate
```

### 3. Install Dependencies

```bash
pip install flask scikit-learn numpy gunicorn
```

---

### 4. Create Files and Folders

#### `train_model.py`

```python
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
import pickle
import os

iris = load_iris()
X, y = iris.data, iris.target

model = RandomForestClassifier()
model.fit(X, y)

os.makedirs("model", exist_ok=True)
with open("model/iris_model.pkl", "wb") as f:
    pickle.dump(model, f)
```

Run once:

```bash
python train_model.py
```

#### `app.py`

```python
import numpy as np
from flask import Flask, request, render_template
import pickle

app = Flask(__name__)
model = pickle.load(open('model/iris_model.pkl', 'rb'))

@app.route('/')
def home():
    return render_template('home.html')

@app.route('/predict', methods=['POST'])
def predict():
    values = [float(x) for x in request.form.values()]
    prediction = model.predict([values])[0]
    return render_template('home.html', prediction_text=f'Iris class predicted: {prediction}')

if __name__ == '__main__':
    app.run(debug=True)
```

#### `templates/home.html`

```html
<!DOCTYPE html>
<html>
<head>
    <title>Iris Prediction</title>
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <style>
        body { font-family: Arial; max-width: 400px; margin: auto; padding: 20px; }
        input { width: 100%; margin-top: 10px; padding: 10px; }
    </style>
</head>
<body>
    <h2>Iris Classifier</h2>
    <form method="POST" action="/predict">
        <input type="text" name="sepal_length" placeholder="Sepal Length" required>
        <input type="text" name="sepal_width" placeholder="Sepal Width" required>
        <input type="text" name="petal_length" placeholder="Petal Length" required>
        <input type="text" name="petal_width" placeholder="Petal Width" required>
        <input type="submit" value="Predict">
    </form>
    {% if prediction_text %}
        <h3>{{ prediction_text }}</h3>
    {% endif %}
</body>
</html>
```

#### `requirements.txt`

```txt
flask
scikit-learn
numpy
gunicorn
```

---

##  Test Locally

```bash
python app.py
```

Open: `http://127.0.0.1:5000`

---

##  Step 1: Push Your App to GitHub

### A. Initialize Git Repo

```bash
git init
git add .
git commit -m "Initial commit for Iris Flask app"
```

### B. Create Repo on GitHub

Go to GitHub → New Repo → Name: `iris-flask-app`

### C. Add Remote & Push

```bash
git remote add origin https://github.com/<your-username>/iris-flask-app.git
git push -u origin main
```

---

##  Step 2: Create Azure Web App

1. Go to [https://portal.azure.com](https://portal.azure.com)
2. Navigate to **App Services** → **Create**
3. Set:

   * Name: `iris-flask-app`
   * Publish: Code
   * Runtime Stack: Python 3.10 or 3.11
   * OS: Linux
   * Region: closest to you
   * Plan: Free (`F1`)

---

##  Step 3: Link GitHub for CI/CD

1. In App Service → **Deployment Center**
2. Choose **GitHub**
3. Select your repo and branch
4. Finish setup — it triggers first deployment

---

##  Step 4: Configure Startup Command

1. App Service → **Configuration** → **General Settings**
2. Set **Startup Command** to:

```
gunicorn app:app
```

---

##  Step 5: Confirm Directory Structure

Make sure repo looks like:

```
iris-flask-app/
│
├── model/
│   └── iris_model.pkl
├── templates/
│   └── home.html
├── app.py
├── requirements.txt
├── README.md
└── .gitignore
```

---

##  Step 6: Test Live App

After successful deployment:

🌐 Visit:

```
https://<your-app-name>.azurewebsites.net
```

---

##  Cleanup & Best Practices

### `.gitignore`

```txt
venv/
__pycache__/
*.pkl
*.log
```

### `README.md`

Explain:

* Project purpose
* How to run locally
* How to deploy

---

##  Notes

* Free tier (`F1`) is good for demo/testing
* You can access the app from **desktop or mobile**
* Azure auto-redeploys on every `git push`
* Gunicorn is required for Azure Linux hosting

---

